In [10]:
import pandas as pd

df = pd.read_csv(r'C:\DEPI project\Smart-Car-New\Data Cleaning\cars_clean.csv')

df.head()

,id,make,model,year,km,price_egp,transmission,color,city,seller_type,fuel_type
0,7284417,Toyota,Yaris,2009,170000.0,540000,Automatic,Dark blue,Heliopolis,dealer,Petrol
1,7296785,Toyota,Camry,2013,113000.0,1000000,Automatic,White,Nasr city,private,Petrol
2,7293331,Toyota,Corolla,2024,95000.0,1380000,Automatic,Black,Mohandessin,private,Petrol
3,7288941,Toyota,Corolla,2008,210000.0,585000,Automatic,Black,Heliopolis,dealer,Petrol
4,7295928,Toyota,Corolla,2022,126000.0,980000,Automatic,Dark red,Maadi,dealer,Petrol


In [11]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 3265 entries, 0 to 3264
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            3265 non-null   int64  
 1   make          3265 non-null   str    
 2   model         3265 non-null   str    
 3   year          3265 non-null   int64  
 4   km            3081 non-null   float64
 5   price_egp     3265 non-null   int64  
 6   transmission  3265 non-null   str    
 7   color         3207 non-null   str    
 8   city          3265 non-null   str    
 9   seller_type   3265 non-null   str    
 10  fuel_type     3265 non-null   str    
dtypes: float64(1), int64(3), str(7)
memory usage: 280.7 KB


id                0
make              0
model             0
year              0
km              184
price_egp         0
transmission      0
color            58
city              0
seller_type       0
fuel_type         0
dtype: int64

In [12]:
df['price_egp'].describe()

count    3.265000e+03
mean     8.939202e+05
std      7.882428e+05
min      5.500000e+04
25%      4.800000e+05
50%      7.000000e+05
75%      1.075000e+06
max      1.080000e+07
Name: price_egp, dtype: float64

In [13]:
df['model'].value_counts()

model
Sunny       352
Corolla     239
Sportage    230
Cerato      167
Qashqai     166
           ... 
Xd            1
Kona          1
H1            1
Ioniq 5       1
Veracruz      1
Name: count, Length: 121, dtype: int64

In [14]:
df['make'].value_counts()

make
Hyundai    1152
Kia         984
Nissan      724
Toyota      405
Name: count, dtype: int64

In [15]:
df.groupby('make')['model'].nunique().sort_values(ascending=False)

make
Hyundai    42
Kia        30
Toyota     30
Nissan     21
Name: model, dtype: int64

In [16]:
df['transmission'].value_counts()

transmission
Automatic    2788
Manual        477
Name: count, dtype: int64

In [17]:
data = df[['make', 'year', 'km', 'transmission', 'model']].copy()

In [18]:
data.isnull().sum()

make              0
year              0
km              184
transmission      0
model             0
dtype: int64

In [19]:
data['model'].value_counts()

model
Sunny       352
Corolla     239
Sportage    230
Cerato      167
Qashqai     166
           ... 
Xd            1
Kona          1
H1            1
Ioniq 5       1
Veracruz      1
Name: count, Length: 121, dtype: int64

In [20]:
X = data[['make', 'year', 'km', 'transmission']]
y = data['model']

In [22]:
data['model'].value_counts().value_counts().sort_index()

count
1      30
2      10
3      10
4       7
5       7
6       2
7       4
8       1
9       3
10      3
11      1
12      2
14      2
17      2
18      1
19      3
21      1
23      1
26      1
27      1
29      2
33      1
38      1
41      2
43      3
48      1
51      1
52      1
62      1
72      1
74      1
76      1
81      2
93      1
100     1
103     1
109     1
123     1
129     1
166     1
167     1
230     1
239     1
352     1
Name: count, dtype: int64

In [23]:
model_counts = df['model'].value_counts()

valid_models = model_counts[model_counts >= 10].index

data = df[df['model'].isin(valid_models)].copy()

In [24]:
X = data[['make', 'year', 'km', 'price_egp', 'transmission']]
y = data['model']

In [25]:
print("Number of models:", y.nunique())
print("Number of cars:", len(data))
print(y.value_counts().tail())

Number of models: 47
Number of cars: 3047
model
Koup         12
X Trail      11
Tiida        10
Seltos       10
Grand I10    10
Name: count, dtype: int64


In [26]:
y.value_counts().min()

np.int64(10)

In [27]:
from sklearn.model_selection import train_test_split

X = data[['make', 'year', 'km', 'price_egp', 'transmission']]
y = data['model']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_cols = ['year', 'km', 'price_egp']
cat_cols = ['make', 'transmission']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [29]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

In [30]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](47,)","['Accent','Accent Rb','Atos',...,'X Trail','Xceed','Yaris']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['make','year','km','price_egp','transmission']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyin

In [31]:
y_pred = model.predict(X_test)

In [32]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy %:", accuracy * 100)

Accuracy: 0.5426229508196722
Accuracy %: 54.26229508196722


In [33]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      Accent       0.43      0.43      0.43        21
   Accent Rb       0.34      0.69      0.46        16
        Atos       0.00      0.00      0.00         3
      Avante       0.00      0.00      0.00         8
      Carens       0.00      0.00      0.00        15
    Carnival       0.00      0.00      0.00         5
      Cerato       0.39      0.88      0.54        33
     Corolla       0.86      1.00      0.92        48
       Creta       0.00      0.00      0.00         3
     Elantra       0.20      0.36      0.26        22
  Elantra Ad       0.00      0.00      0.00         9
 Elantra Cn7       0.10      0.11      0.11         9
  Elantra Hd       0.00      0.00      0.00        10
  Elantra Md       0.00      0.00      0.00         8
       Excel       0.58      0.70      0.64        10
    Fortuner       1.00      1.00      1.00         6
        Getz       0.00      0.00      0.00         5
Grand Cerato       0.00    

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag